In [ ]:
import trimesh, random
from tqdm.notebook import tqdm
import numpy as np
import fs.osfs
import fs.zipfs
import fs.path
import glob
import pathlib

In [ ]:
import matplotlib.pyplot as plt


def display_stats(a, title):
    _ = plt.hist(np.minimum(a, 64), bins="auto")  # arguments are passed to np.histogram
    plt.title(title)
    # plt.xlabel("Collision mesh count")
    # plt.ylabel("# objects with this collision mesh count")
    plt.show()
    print("Median", np.median(a))
    print("Mean", np.mean(a))
    print("Std", np.std(a))
    print("Max", np.max(a))
    print("Min", np.min(a))

In [ ]:
# dataset = fs.zipfs.ZipFS(r"D:\BEHAVIOR-1K\asset_pipeline\artifacts\og_dataset.zip")
# dataset = fs.zipfs.ZipFS(r"D:\BEHAVIOR-1K\asset_pipeline\artifacts\parallels\vertex_reduction.zip")
dataset = fs.osfs.OSFS(r"D:\BEHAVIOR-1K\asset_pipeline\artifacts\aggregate")

In [ ]:
collision_objs = [x.path for x in dataset.glob(r"objects/*/*/shape/collision/*.obj")]
# collision_objs = glob.glob(r"D:\BEHAVIOR-1K\asset_pipeline\artifacts\aggregate\objects\*\*\shape\collision\*.obj")
# collision_objs = glob.glob(r"C:\Users\Cem\Downloads\redux\*\*\shape\collision\*.obj")
print(len(collision_objs))
random.shuffle(collision_objs)

## Count Collision Hulls

In [ ]:
import io
import trimesh

# Count the number of o-directives in each
o_directives = {}
failed = []
for p in tqdm(collision_objs):
    parts = fs.path.parts(p)
    target = parts[-5] + "-" + parts[-4]

    with dataset.open(p, "rb") as f:
        bio = io.BytesIO(f.read())

    m = trimesh.load(bio, file_type="obj", force="mesh", skip_material=True)
    submeshes = m.split(only_watertight=False)
    for s in submeshes:
        try:
            trimesh.convex.convex_hull(s)
        except:
            failed.append(target)

    # for submesh in submeshes:
    #     if len(submesh.vertices) > 60:
    #         print(p, "has mesh with", len(submesh.vertices), "vertices")
    #         failed.append(target)
    o_directives[target] = len(submeshes)

In [ ]:
print(set(failed))

In [ ]:
for k, v in o_directives.items():
    if v > 100:
        print(k, "has", v, "hulls")

In [ ]:
a = np.array(list(o_directives.values()))
display_stats(a, "Collision mesh count histogram")

## Analyze bounding box volumes

In [ ]:
col_vols = {}
vis_vols = {}
bb_volume_ratios = {}
bb_max_nonzero_dim_ratios = {}
filtered_objs = collision_objs  # [x for x in collision_objs if "floors" in x]
watertight = 0
for col_name in tqdm(filtered_objs):
    col_name = pathlib.Path(col_name)
    target = col_name.parts[-5] + "-" + col_name.parts[-4]
    vis_name = pathlib.Path(str(col_name).replace("collision", "visual"))
    col_mesh = trimesh.load(col_name, force="mesh", skip_material=True)
    vis_mesh = trimesh.load(
        vis_name, force="mesh", skip_material=True, merge_tex=True, merge_norm=True
    )
    if vis_mesh.is_volume:
        watertight += 1

    col_bb = col_mesh.bounding_box.extents
    vis_bb = vis_mesh.bounding_box.extents

    col_vol = np.product(col_bb)
    vis_vol = np.product(vis_bb)

    col_vols[target] = col_vol
    vis_vols[target] = vis_vol
    bb_volume_ratios[target] = col_vol / vis_vol

    dim_ratio = col_bb / vis_bb
    invalid_dims = np.isclose(vis_bb, 0, atol=1e-3)
    if np.all(invalid_dims):
        print(
            f"Error: {target} bb's all dimensions are close to zero, can't include it:",
            vis_bb,
        )
        continue
    elif np.any(invalid_dims):
        print(f"Warning: {target} bb has dimension close to zero:", vis_bb)
    bb_max_nonzero_dim_ratios[target] = np.max(dim_ratio[~invalid_dims])

In [ ]:
print("Total", len(filtered_objs))
print("Watertight", watertight)
print("Ratio", watertight / len(filtered_objs))

In [ ]:
a = np.array(list(col_vols.values()))
display_stats(np.log(a), "Collision bounding box volume histogram")

In [ ]:
a = np.array(list(vis_vols.values()))
display_stats(np.log(a), "Visual bounding box volume histogram")

In [ ]:
a = np.array(list(bb_volume_ratios.values()))
display_stats(a, "Collision-to-visual bounding box volume ratio histogram")

In [ ]:
a = np.array(list(bb_max_nonzero_dim_ratios.values()))
display_stats(
    a, "Maximum collision-to-visual bounding box non-zero dimension ratio histogram"
)

## Objects with too many links

In [ ]:
import sys

sys.path.append(r"D:\BEHAVIOR-1K\asset_pipeline")
from b1k_pipeline.utils import get_targets, PIPELINE_ROOT, parse_name

In [ ]:
import json
from collections import defaultdict

# Count the number of objects with non-tagged fixed links
link_cnts = {}
for target in tqdm.tqdm(get_targets("combined")):
    obj_list_file = PIPELINE_ROOT / "cad" / target / "artifacts" / "object_list.json"
    assert obj_list_file.exists(), "Missing obj list file " + str(obj_list_file)
    with open(obj_list_file, "r") as f:
        data = json.load(f)
        meshes = data["meshes"]
        parsed = [parse_name(x) for x in meshes]
        obj_link_cnts = defaultdict(set)
        for p in parsed:
            if not p.group("link_name") or p.group("link_name") == "base_link":
                continue
            if p.group("bad"):
                continue
            obj_name = p.group("obj_basename")
            link_name = p.group("link_basename")
            # Don't count openable links
            if p.group("tag") and "openable" in p.group("tag"):
                continue
            obj_link_cnts[obj_name].add(link_name)
        # fixed = sum(1 for x in parsed if x is not None and x.group("joint_type") == "F" and not x.group("tag"))
        link_cnts.update({x: len(y) for x, y in obj_link_cnts.items()})

In [ ]:
interesting = np.asarray(list(link_cnts.values()), dtype=int)
interesting = interesting[interesting > 3]
display_stats(a, "Collision-to-visual bounding box volume ratio histogram")

In [ ]:
print("\n".join(str(x) for x in sorted(link_cnts.items(), key=lambda x: -x[1])))

In [ ]:
m = trimesh.load(
    r"C:\Users\Cem\Downloads\test-fridge\alarm-dkwmmf-base_link.obj",
    force="mesh",
    skip_material=True,
    merge_tex=True,
    merge_norm=True,
)

In [ ]:
len(m.split())